# Property Listing Content Generator — Eval-First

The deliverable here is a **validated evaluator**, not the generator. This notebook runs **fully offline (zero API calls)**: it re-executes the deterministic evaluation live and reads the LLM-judged results from committed `.eval` logs.

Two clearly-separated kinds of result below:
1. **Committed real runs** — genuine Anthropic generations + judge scores, read from `logs/`.
2. **Live offline recompute** — deterministic scorers re-run here via a replay solver, proving reproducibility without a key.

View the raw logs interactively with: `uv run inspect view --log-dir logs`

In [1]:
import json
from content_pipeline.report import find_log, gold_report

def show_metrics(log):
    for s in log.results.scores:
        print(f"  {s.name:24s} " + ", ".join(f"{k}={v.value:.2f}" for k, v in s.metrics.items()))

## 1. A generated listing (committed real run)
Structured, grounded copy. Note each highlight carries the `source_field` it derives from — that binding makes citation grounding a deterministic check.

In [2]:
print(json.dumps(json.load(open('fixtures/generations/villa.json')), indent=2))

{
  "hero_headline": "Seafront villa with private pool steps from Sitges marina",
  "highlights": [
    {
      "text": "Three-bedroom villa sleeping up to 6 guests with 2 bathrooms",
      "source_field": "rental_info"
    },
    {
      "text": "Private pool and stunning sea views",
      "source_field": "amenities"
    },
    {
      "text": "Ten-minute walk to the old town",
      "source_field": "reviews"
    },
    {
      "text": "Rated 4.96 out of 5 by 87 guests",
      "source_field": "average_review_score"
    },
    {
      "text": "Steps from the marina in Sitges, Spain",
      "source_field": "location"
    }
  ],
  "about_section": "Casa del Mar is a bright three-bedroom villa located steps from the marina in Sitges, Spain. This seafront property accommodates up to 6 guests across 3 bedrooms and 2 bathrooms, with a private pool and sea views. Guests can walk to the old town in ten minutes, making it an ideal base for exploring the area.",
  "amenity_descriptions": [
    {

## 2. Production scorers on the real generations (committed)
Tier 1 deterministic (citation validity, quality) + Tier 2 LLM judge (atomic-claim faithfulness, brand voice). These numbers come from the committed `generation_eval` log.

In [3]:
gen = find_log('generation_eval', with_judge=True)
print('generation_eval (real Anthropic run):')
show_metrics(gen)

generation_eval (real Anthropic run):
  citation_validity        accuracy=1.00, stderr=0.00
  quality                  accuracy=1.00, stderr=0.00
  faithfulness             mean=0.96, stderr=0.04
  model_graded_qa          accuracy=0.75, stderr=0.25


## 3. META-EVAL — validating the evaluator (the point of the submission)
A suite that scores everything "pass" is worthless. We prove the scorers (a) **catch real failures** (recall) and (b) **don't cry wolf** on good copy (false-positive rate), over hand-labeled good/bad generations.

### 3a. Deterministic tier — recomputed LIVE, zero API
Re-run here via the replay solver + `mockllm` (no key). These match the committed deterministic scores bit-for-bit.

In [4]:
import tempfile
from inspect_ai import eval as inspect_eval
from content_pipeline.eval_task import meta_eval

with tempfile.TemporaryDirectory() as d:
    live = inspect_eval(meta_eval(include_judge=False), model='mockllm/model', log_dir=d, display='none')[0]
print('meta_eval deterministic (live, offline):')
show_metrics(live)

Output()

meta_eval deterministic (live, offline):
  meta_citation_validity   detection_rate=1.00, false_positive_rate=0.00
  meta_quality             detection_rate=1.00, false_positive_rate=0.00


### 3b. Judge tier — detection on hallucinated negatives (committed)
The judge needs a key, so its verdicts are read from the committed `meta_eval` log. Faithfulness should be low on the planted hallucinations (fabricated landmark, wrong guest count) and high on good copy.

In [5]:
meta = find_log('meta_eval', with_judge=True)
print(f"{'sample':26s} {'label':5s} {'judge_target':12s} faithfulness")
for s in sorted(meta.samples, key=lambda s: s.id):
    sc = s.scores.get('faithfulness')
    if sc:
        tag = 'hallucination' if s.metadata['judge_failure'] else ''
        print(f"  {s.id:26s} {s.metadata['label']:5s} {tag:12s} {sc.value:.2f}")

sample                     label judge_target faithfulness
  bad_bland                  bad                0.25
  bad_empty_citation         bad                0.83
  bad_fabricated_landmark    bad   hallucination 0.50
  bad_unlisted_amenity       bad                0.57
  bad_wrong_number           bad   hallucination 0.50
  good_cottage               good               0.67
  good_villa                 good               0.89


### 3c. Judge vs human — confusion matrix (committed verdicts vs gold labels)
Positive class = *not grounded* (a hallucination the judge should flag). **Read this as methodology, not a trustworthy point estimate** — n is tiny and there is a single annotator (see caveats).

In [6]:
r = gold_report()
print(f"n={len(r.rows)}   agreement={r.agreement:.0%}   recall={r.recall:.0%}   precision={r.precision:.0%}\n")
print('                 judge:flag   judge:ground')
print(f"human:not-ground   TP={r.tp:<8d}   FN={r.fn}")
print(f"human:grounded     FP={r.fp:<8d}   TN={r.tn}\n")
print('Disagreements (judge over-flagged legitimate claims):')
for row in r.rows:
    if not row.agree:
        print(f"  judge={row.judge_verdict!r}: {row.claim}")

n=13   agreement=85%   recall=100%   precision=71%

                 judge:flag   judge:ground
human:not-ground   TP=5          FN=0
human:grounded     FP=2          TN=6

Disagreements (judge over-flagged legitimate claims):
  judge='unsupported': Stay connected with high-speed broadband internet access throughout the property.
  judge='unsupported': Air conditioning throughout for warm summers.


## 4. Reading
- **Deterministic tier**: perfect detection / zero false positives on this set — but it only covers citation validity + structural quality (cheap pre-filter).
- **Judge tier**: catches every planted hallucination (recall 100%) including a prompt-injection attempt, but **over-flags** ~2 legitimate claims ("high-speed broadband", "air conditioning throughout") — a real calibration finding, not hidden.
- The eval also caught a **bug in my own code**: an incomplete grounding corpus made the judge mark valid claims unsupported (good-copy faithfulness 0.77). Fixing the corpus moved it to 0.96. That is the eval-first loop working.

### Honest caveats (also in README)
- n is a demonstration size; treat per-check rates as methodology that scales to hundreds, not significant estimates.
- Single annotator who also wrote the judge prompt — no inter-annotator agreement.
- The gold set validates judge *verdicts*, not the upstream claim *decomposition*.
- Deferred for time: omission/coverage (did it sell the best true facts), multilingual, image grounding, regression gating in CI.